# LLM Text Preprocessing Foundations (Embeddings)

This notebook walks through the full text preprocessing pipeline for a Large Language Model, 
based on Chapter 2 of *Build a Large Language Model From Scratch* by Sebastian Raschka.

In [ ]:
# install required libraries
%pip install torch tiktoken

In [2]:
import torch
import tiktoken

## Step 1: Loading and Tokenizing Text

> *Source: ch02.ipynb — Section 2.2 (Tokenizing text) & Section 2.5 (BytePair encoding)*

### Why tokenization matters for LLMs and agentic systems

Tokenization is the critical first step in any NLP pipeline. An LLM cannot process raw strings; it needs a way to break continuous text into discrete units (tokens) that can be mapped to numbers. Without tokenization, there is no way to build a vocabulary or feed text into a neural network.

For agentic systems, tokenization determines the granularity at which the model "sees" language. A poorly chosen tokenization scheme can waste context window space, misrepresent rare words, or fail on multilingual input; all of which degrade an agent's ability to reason and act. BPE (Byte Pair Encoding), used by GPT-2, strikes a balance: common words stay as single tokens, while rare/unknown words are split into meaningful subword pieces, so the model never encounters a truly "unknown" token.

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of characters:", len(raw_text))
print("First 100 chars:", raw_text[:100])

Total number of characters: 20479
First 100 chars: I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


---

## Step 2: BPE Tokenization with tiktoken

> *Source: ch02.ipynb — Section 2.5 (BytePair encoding)*

### Why BPE matters for LLMs and agentic systems

BPE (Byte Pair Encoding) is the tokenizer used by GPT-2. It is very smart because it does not just split by spaces or punctuation. Instead, it breaks words into smaller pieces called "subwords". This means that even if the model has never seen a word before, it can still understand it by breaking it into parts it already knows. For example, the word "unfamiliarword" could be split into ["unfam", "iliar", "word"].

This is super important for agentic systems because an agent might receive any kind of input (like code, new words, or even different languages). BPE makes sure the model can always handle the input without crashing or giving an "unknown token" error. It keeps the vocabulary size manageable while still being able to represent any text.

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print("Encoded token IDs:", integers)

strings = tokenizer.decode(integers)
print("Decoded back to text:", strings)

Encoded token IDs: [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Decoded back to text: Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


---

## Step 3: Data Sampling with a Sliding Window

> *Source: ch02.ipynb — Section 2.6 (Data sampling with a sliding window)*

### Why sliding window sampling matters for LLMs

LLMs learn to predict the next word. So we need to prepare the training data in a way that the model gets an input sequence and a target sequence where the target is shifted by one position. For example, if the input is `[A, B, C, D]`, the target is `[B, C, D, E]`.

We use a "sliding window" to create many of these input-target pairs from a single text. The two important parameters are:
- **max_length**: how many tokens each sample has (this is the context window size)
- **stride**: how many positions we move the window each time

If stride is smaller than max_length, the windows will overlap. This means some tokens appear in multiple training samples, which gives the model more chances to learn from those tokens. This is useful when we have a small dataset because it creates more training samples.

In [ ]:
enc_text = tokenizer.encode(raw_text)
print("Total tokens in the text:", len(enc_text))

Total tokens in the text: 5145


In [ ]:
# Section 2.6 (input-target pairs with sliding window)
enc_sample = enc_text[50:]
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

print("\nStep by step predictions:")
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]

Step by step predictions:
 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [ ]:
# Section 2.6 (GPTDatasetV1 class & create_dataloader_v1 function)
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
# Section 2.6 (testing the dataloader)
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print("First batch:", first_batch)

second_batch = next(data_iter)
print("Second batch:", second_batch)

First batch: [tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
Second batch: [tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [ ]:
# Section 2.6 (batched outputs with stride=max_length)
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


---

## Step 4: Creating Token Embeddings

> *Source: ch02.ipynb — Section 2.7 (Creating token embeddings)*

### Why do embeddings encode meaning, and how are they related to NN concepts?

This is one of the most interesting parts. An embedding is a way to turn a token ID (which is just a number) into a vector (a list of numbers). But the thing is that these vectors are not random. During training, the neural network learns to put similar words close together in this vector space. For example, "cat" and "dog" would end up with similar vectors because they appear in similar contexts.

This works because of how neural networks learn. The embedding layer is basically a big table (a weight matrix) where each row is the vector for one token. When the model trains with backpropagation, these vectors get adjusted so that the model can predict the next word better. So over time, words that are used in similar ways get similar vectors. That is how embeddings "encode meaning" - the meaning comes from the patterns the NN finds in the training data.

In NN terms, the embedding layer is just a lookup table that is equivalent to doing one-hot encoding followed by a matrix multiplication with a linear layer, but much more efficient. The weights of this layer are learned during training, just like any other layer in the network.

In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print("Embedding weight matrix:")
print(embedding_layer.weight)

print("\nEmbedding for token ID 3:")
print(embedding_layer(torch.tensor([3])))

print("\nEmbeddings for all input IDs [2, 3, 5, 1]:")
print(embedding_layer(input_ids))

Embedding weight matrix:
Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)

Embedding for token ID 3:
tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)

Embeddings for all input IDs [2, 3, 5, 1]:
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


---

## Step 5: Encoding Word Positions

> *Source: ch02.ipynb — Section 2.8 (Encoding word positions)*

### Why positional embeddings are needed

There is a problem with token embeddings alone: the same word always gets the same vector, no matter where it is in the sentence. But word order matters a lot! "The cat sat on the mat" is very different from "The mat sat on the cat". 

To fix this, we add **positional embeddings**. These are another set of learned vectors, one for each position in the context window. We just add them to the token embeddings. This way, the model knows not only *what* each token is, but also *where* it is in the sequence. GPT-2 uses absolute positional embeddings, which means position 0 always has the same positional vector, position 1 always has the same one, etc.

In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Token IDs:\n", inputs)
print("\nInputs shape:", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape: torch.Size([8, 4])


In [ ]:
# Section 2.8 (token + positional embeddings)
token_embeddings = token_embedding_layer(inputs)
print("Token embeddings shape:", token_embeddings.shape)

context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print("Positional embeddings shape:", pos_embeddings.shape)

input_embeddings = token_embeddings + pos_embeddings
print("Final input embeddings shape:", input_embeddings.shape)

Token embeddings shape: torch.Size([8, 4, 256])
Positional embeddings shape: torch.Size([4, 256])
Final input embeddings shape: torch.Size([8, 4, 256])


---

## Experiment: Changing max_length and stride

> *Based on the GPTDatasetV1 class and create_dataloader_v1 function from ch02.ipynb — Section 2.6*

In this experiment, I want to see how changing `max_length` and `stride` affects the number of training samples we get from the same text. This is important because these two parameters control how many input-target pairs the model will learn from.

- **max_length** = the size of each training sample (how many tokens the model sees at once)
- **stride** = how many tokens we skip before starting the next sample

I will test several combinations and report the number of samples for each one. I will also explain why overlap (when stride < max_length) can be useful for training.

In [ ]:
total_tokens = len(tokenizer.encode(raw_text))
print(f"Total tokens in 'the-verdict.txt': {total_tokens}\n")

experiments = [
    {"max_length": 4,   "stride": 4},    # no overlap
    {"max_length": 4,   "stride": 1},    # big overlap (stride=1)
    {"max_length": 4,   "stride": 2},    # some overlap
    {"max_length": 8,   "stride": 8},    # no overlap, bigger context
    {"max_length": 8,   "stride": 4},    # 50% overlap, bigger context
    {"max_length": 16,  "stride": 16},   # no overlap, even bigger context
    {"max_length": 16,  "stride": 4},    # big overlap, big context
    {"max_length": 256, "stride": 256},  # no overlap, large context (like GPT-2 style)
    {"max_length": 256, "stride": 128},  # 50% overlap, large context
]

print(f"{'max_length':>12} | {'stride':>8} | {'samples':>10} | {'overlap?':>10}")
print("-" * 50)

for exp in experiments:
    ml = exp["max_length"]
    st = exp["stride"]
    num_samples = len(range(0, total_tokens - ml, st))
    has_overlap = "Yes" if st < ml else "No"
    print(f"{ml:>12} | {st:>8} | {num_samples:>10} | {has_overlap:>10}")

Total tokens in 'the-verdict.txt': 5145

  max_length |   stride |    samples |   overlap?
--------------------------------------------------
           4 |        4 |       1286 |         No
           4 |        1 |       5141 |        Yes
           4 |        2 |       2571 |        Yes
           8 |        8 |        643 |         No
           8 |        4 |       1285 |        Yes
          16 |       16 |        321 |         No
          16 |        4 |       1283 |        Yes
         256 |      256 |         20 |         No
         256 |      128 |         39 |        Yes


In [ ]:
# Case 1: max_length=4, stride=4 (NO overlap)
dataset_no_overlap = GPTDatasetV1(raw_text, tokenizer, max_length=4, stride=4)
print(f"max_length=4, stride=4 (no overlap): {len(dataset_no_overlap)} samples")

# Case 2: max_length=4, stride=1 (WITH overlap)
dataset_with_overlap = GPTDatasetV1(raw_text, tokenizer, max_length=4, stride=1)
print(f"max_length=4, stride=1 (with overlap): {len(dataset_with_overlap)} samples")

print(f"\nWith overlap we get {len(dataset_with_overlap) - len(dataset_no_overlap)} more samples from the same text")
print(f"That is {len(dataset_with_overlap) / len(dataset_no_overlap):.1f}x more training data.")

max_length=4, stride=4 (no overlap): 1286 samples
max_length=4, stride=1 (with overlap): 5141 samples

With overlap we get 3855 MORE samples from the same text!
That is 4.0x more training data.


In [ ]:
# Case 3: max_length=256, stride=256 (NO overlap)
dataset_big_no_overlap = GPTDatasetV1(raw_text, tokenizer, max_length=256, stride=256)
print(f"max_length=256, stride=256 (no overlap): {len(dataset_big_no_overlap)} samples")

# Case 4: max_length=256, stride=128 (50% overlap)
dataset_big_overlap = GPTDatasetV1(raw_text, tokenizer, max_length=256, stride=128)
print(f"max_length=256, stride=128 (50% overlap): {len(dataset_big_overlap)} samples")

print(f"\nWith 50% overlap we get {len(dataset_big_overlap) - len(dataset_big_no_overlap)} more samples.")
print(f"That is about {len(dataset_big_overlap) / len(dataset_big_no_overlap):.1f}x more training data.")

max_length=256, stride=256 (no overlap): 20 samples
max_length=256, stride=128 (50% overlap): 39 samples

With 50% overlap we get 19 more samples.
That is about 1.9x more training data.


### Experiment Results and Analysis

From the experiment above, we can see some clear patterns:

**Effect of stride:**
- When stride = max_length (no overlap), we get the fewest samples because each token only appears in one training sample.
- When stride < max_length (with overlap), we get many more samples. For example, with max_length=4 and stride=1, we get about 4x more samples than with stride=4. This is because the sliding window moves only 1 token at a time, so most tokens appear in multiple samples.

**Effect of max_length:**
- A bigger max_length means each sample contains more tokens, so the model sees more context at once. But it also means we get fewer total samples from the same text, because each "window" covers more of the text.
- With max_length=256, we get way fewer samples than with max_length=4. This makes sense because each sample is 64 times longer.

**Why is overlap useful?**
- Overlap is useful because it creates more training samples from the same data. This is especially helpful when we have a small dataset (like our short story).
- With overlap, the model sees the same tokens in different positions and contexts, which helps it learn better patterns.
- However, too much overlap can lead to overfitting because the model keeps seeing almost the same data repeated. The book mentions this trade-off: we need to balance getting enough training samples with not repeating too much.
- In practice, a 50% overlap (stride = max_length / 2) is a great choice because it gives a good balance between more data and not too much repetition.